In [1]:
import json
from pathlib import Path
import pandas as pd
import re
import extract as ext
import pdfplumber
import extract_exercises_from_exam_file as parse
from random import choice

In [2]:
ROOT = Path("~/code/selector_examenes").expanduser()

In [3]:
def load_keywords(json_file=ROOT / "json/quimica_keywords.json"):
    with open(json_file) as file:
        return json.load(file)

In [4]:
def classify_exercise(statement: str, keywords = None) -> list:
    if keywords is None:
        keywords = load_keywords()

    # convertir saltos de línea en espacios
    statement = re.sub(r"\s+", " " , statement)

    
    # Vocales acentuadas (minúsculas y mayúsculas) → sin acentuar
    tabla = str.maketrans(
        "áéíóúàèìòùäëïöüâêîôûÁÉÍÓÚÀÈÌÒÙÄËÏÖÜÂÊÎÔÛ",
        "aeiouaeiouaeiouaeiouAEIOUAEIOUAEIOUAEIOU"
        )

    statement = statement.translate(tabla)
    topics = {}
    for topic in keywords.keys():
        topic_keywords = keywords[topic]["topic_kw"]
        pattern = re.compile("|".join(topic_keywords), flags=re.IGNORECASE | re.DOTALL)
        keywords_match_count = re.findall(pattern, statement)
        if len(keywords_match_count) > 0:
            topics[topic] = len(keywords_match_count)

    most_occurrences_topic = [k for k,v in topics.items() if v == max(topics.values())]

    return most_occurrences_topic

In [5]:
def create_df_exercises_from_exam(exam_file: Path) -> pd.DataFrame:
    exercises_list = []

    # extraer el texto del examen
    exercises_text = parse.extract_exercises_from_pdf(exam_file)

    # clasificar el periodo del examen
    year = int(re.search(r"\d{4}", exam_file.name)[0])
    exam_period = parse.get_exam_period(year)

    # anadir los ejercicios a una lista de diccionarios para
    # luego crear el dataframe
    for ex in parse.split_exercises(exercises_text, exam_period):
        topic = classify_exercise(statement=ex)
        exercises_list.append({"year" : year, "exam" : exam_file.name, "topic" : topic, "statement" : ex})
              
    return pd.DataFrame(exercises_list)

Clasificar ejercicios de un archivo

In [6]:
file = ROOT / "examenes_quimica/2025_quimica/2025_quimica_Quimica - Examen_Titular-A_Quimica.pdf"

In [7]:
create_df_exercises_from_exam(file).explode("topic")

,year,exam,topic,statement
0,2025,2025_quimica_Quimica - Examen_Titular-A_Quimic...,configuracion_electronica,1A. Escriba las configuraciones electrónicas d...
1,2025,2025_quimica_Quimica - Examen_Titular-A_Quimic...,enlace_quimico,"1B. a) Dadas las moléculas H₂S y PF₃, razone e..."
2,2025,2025_quimica_Quimica - Examen_Titular-A_Quimic...,termoquimica,2A. Justifique si son verdaderas o falsas las ...
3,2025,2025_quimica_Quimica - Examen_Titular-A_Quimic...,cinetica,2B. La reacción química 2A + B → C tiene como ...
4,2025,2025_quimica_Quimica - Examen_Titular-A_Quimic...,equilibrio,3A. El equilibrio de descomposición del NaHCO₃...
5,2025,2025_quimica_Quimica - Examen_Titular-A_Quimic...,acido-base,3B. Se preparan 250 mL de una disolución acuos...
6,2025,2025_quimica_Quimica - Examen_Titular-A_Quimic...,formulacion,4A. Nombre o formule los siguientes compuestos...
7,2025,2025_quimica_Quimica - Examen_Titular-A_Quimic...,reactividad_organica,4B. a) Escriba y ajuste la reacción de combust...
8,2025,2025_quimica_Quimica - Examen_Titular-A_Quimic...,redox,PROTECCIÓN CONTRA LA CORROSIÓN\nEl deterioro c...


In [8]:
exam_files = list((ROOT / "examenes_quimica").rglob("*.pdf"))


In [ ]:
# for _ in range(20):
#     random_exam = choice(exam_files)
#     exam_files.remove(random_exam)
#     df = create_df_exercises_from_exam(random_exam).explode("topic")
#     if df.topic.isna().sum() > 0:
#         print(random_exam.name)
#         print(df[df["topic"].isna()].statement.values[0])

In [ ]:
# df_clasificados = pd.DataFrame()
# for file in exam_files:
#     df = create_df_exercises_from_exam(file).explode("topic")
#     df_clasificados = pd.concat([df_clasificados, df], axis=0)

# df = df_clasificados

In [ ]:
# def get_exam_option(row):
#     if row.year <= 2019:
#         exercise_number = row.statement[:21]
#         statement = row.statement[22:].strip()
#     else:
#         exercise_number = row.statement[:2]
#         statement = row.statement[3:].strip()
#     return exercise_number, statement

In [ ]:
# def get_convocatoria(exam):
#     if exam.startswith("química") or exam.startswith("quimica"):
#         return exam.split(" - ")[-1].replace(".pdf", "").title()
#     else:
#         return exam

In [ ]:
# df["exercise_number"], df["statement"] = zip(*df.apply(get_exam_option, axis=1))
# df["convocatoria"] = df.exam.apply(get_convocatoria)

In [ ]:
# df.to_csv("examenes_oficiales_clasificados.csv", index = False)

In [37]:
df = pd.read_csv("examenes_oficiales_clasificados.csv")

In [38]:
df.sample(5)

,year,exam,topic,statement,exercise_number,convocatoria
350,2022,química - 2022 - julio.pdf,reactividad_organica,a) Escriba dos compuestos isómeros de fórmula ...,B6,Julio
635,2025,2025_quimica_Quimica - Examen_Suplente2-B_Quim...,formulacion,Nombre o formule los siguientes compuestos:\na...,4A,2025_quimica_Quimica - Examen_Suplente2-B_Quim...
945,2023,química - 2023 - reserva 2.pdf,reactividad_organica,Considere los siguientes tipos de compuestos o...,B4,Reserva 2
564,2010,química - 2010 - reserva 2.pdf,configuracion_electronica,a) Justifique cómo es el tamaño de un átomo co...,"Ejercicio 2, opción A",Reserva 2
868,2014,química - 2014 - reserva 3.pdf,redox,"Se hace pasar durante 2,5 horas una corriente ...","Ejercicio 6, opción B",Reserva 3


In [39]:
df.isna().sum()

year               0
exam               0
topic              6
statement          0
exercise_number    0
convocatoria       0
dtype: int64

In [40]:
df[df.topic.isna()]

,year,exam,topic,statement,exercise_number,convocatoria
149,2016,química - 2016 - reserva 3.pdf,NaN,En un matraz cerrado de 5 L hay 42 g de N₂ a 2...,"Ejercicio 2, opción A",Reserva 3
414,2018,química - 2018 - septiembre.pdf,NaN,"Dados los siguientes reactivos HI, I₂, H₂/cata...","Ejercicio 4, opción A",Septiembre
534,2012,quimica - 2012 - reserva 1.pdf,NaN,Se preparan 25 mL de una disolución 2’5M de Fe...,"Ejercicio 5, opción A",Reserva 1
610,2010,química - 2010 - reserva 1.pdf,NaN,Indique los reactivos adecuados para realizar ...,"Ejercicio 4, opción A",Reserva 1
715,2024,química - 2024 - julio.pdf,NaN,Escriba las siguientes reacciones:\na) CH₃CH=C...,B6,Julio
836,2014,química - 2014 - reserva 4.pdf,NaN,Justifique la veracidad o falsedad de las sigu...,"Ejercicio 2, opción A",Reserva 4


In [41]:
df.topic.value_counts().sort_values(ascending=False)

topic
acido-base                   219
formulacion                  145
redox                        119
equilibrio                   106
configuracion_electronica     88
reactividad_organica          84
enlace_quimico                82
solubilidad                   70
termoquimica                  45
propiedades_periodicas        28
cantidad_quimica              25
cinetica                      18
Name: count, dtype: int64